In [1]:
import pandas as pd
import numpy as np

In [2]:
import numpy as np
import random

SEED = 2026

np.random.seed(SEED)
random.seed(SEED)

In [3]:
products = pd.read_csv('/content/combined_products_UPDATED.csv')

In [4]:
suppliers = pd.read_csv('/content/synthetic_suppliers.csv')

In [5]:
products.loc[lambda x: x['product'] == 'power_devices']

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
9780,2012-06-01,USA,FRED,USA,99.797898,0.578828,power_devices,2012,6
9781,2012-06-01,JPN,FRED,OAC,100.000000,0.618841,power_devices,2012,6
9782,2012-06-01,DEU,FRED,USA,99.797898,0.598787,power_devices,2012,6
9783,2012-06-01,FRA,FRED,USA,99.797898,0.588808,power_devices,2012,6
9784,2012-06-01,GBR,FRED,USA,99.797898,0.578828,power_devices,2012,6
...,...,...,...,...,...,...,...,...,...
13035,2025-12-01,BRA,FRED,OAC,98.700000,0.500652,power_devices,2025,12
13036,2025-12-01,IND,FRED,OAC,98.700000,0.450587,power_devices,2025,12
13037,2025-12-01,THA,FRED,OAC,98.700000,0.440574,power_devices,2025,12
13038,2025-12-01,IDN,FRED,OAC,98.700000,0.430561,power_devices,2025,12


In [6]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability
0,ARE,SUP_ARE_1,127.022009,9.676976,93.643856,0.245273,0.661656,0.825186
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894
...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959


In [7]:
suppliers.country_code.unique()

array(['ARE', 'AUS', 'BEL', 'BRA', 'CAN', 'CHN', 'DEU', 'FIN', 'FRA',
       'GBR', 'HKG', 'IDN', 'IND', 'JPN', 'MEX', 'MYS', 'NLD', 'SGP',
       'THA', 'USA'], dtype=object)

In [8]:
products['product'].unique()

array(['integrated_circuit_components', 'microprocessors', 'transistors',
       'power_devices'], dtype=object)

# Attaching Products Randomly to Suppliers by Weighted-Tier Levels

In [9]:

# Country-Specific, Industry-Aligned Product Supply Weights


country_product_weights = {
# country_product_weights represents country-product manufacturing strength.
# It is used as a unified proxy for:
# 1) probability that a supplier from that country participates in the product category,
# 2) expected manufacturing alignment / quality performance,
# 3) expected commercial flexibility (discounting and Minimum Order Quantity (MOQ)).

    # -------------------------
    # Tier 1A — IC Component & Semiconductor Powerhouses
    # -------------------------
    "CHN": {
        "Microprocessors": 0.08,
        "Power Devices": 0.22,
        "Transistors": 0.25,
        "IC Components": 0.45
    },
    "MYS": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.20,
        "IC Components": 0.50
    },
    "THA": {
        "Microprocessors": 0.05,
        "Power Devices": 0.25,
        "Transistors": 0.20,
        "IC Components": 0.50
    },
    "SGP": {
        "Microprocessors": 0.15,
        "Power Devices": 0.20,
        "Transistors": 0.15,
        "IC Components": 0.50
    },

    # -------------------------
    # Tier 1B — Advanced Logic / Specialty Analog
    # -------------------------
    "USA": {
        "Microprocessors": 0.30,
        "Power Devices": 0.20,
        "Transistors": 0.25,
        "IC Components": 0.25
    },
    "JPN": {
        "Microprocessors": 0.20,
        "Power Devices": 0.35,
        "Transistors": 0.20,
        "IC Components": 0.25
    },
    "DEU": {
        "Microprocessors": 0.10,
        "Power Devices": 0.45,
        "Transistors": 0.25,
        "IC Components": 0.20
    },

    # -------------------------
    # Tier 2 — Moderate Semiconductor Presence
    # -------------------------
    "MEX": {
        "Microprocessors": 0.10,
        "Power Devices": 0.25,
        "Transistors": 0.30,
        "IC Components": 0.35
    },
    "NLD": {
        "Microprocessors": 0.20,
        "Power Devices": 0.25,
        "Transistors": 0.25,
        "IC Components": 0.30
    },
    "CAN": {
        "Microprocessors": 0.20,
        "Power Devices": 0.05,
        "Transistors": 0.35,
        "IC Components": 0.40
    },
    "FRA": {
        "Microprocessors": 0.15,
        "Power Devices": 0.30,
        "Transistors": 0.25,
        "IC Components": 0.30
    },
    "GBR": {
        "Microprocessors": 0.20,
        "Power Devices": 0.20,
        "Transistors": 0.30,
        "IC Components": 0.30
    },
    "IND": {
        "Microprocessors": 0.10,
        "Power Devices": 0.25,
        "Transistors": 0.30,
        "IC Components": 0.35
    },
    "HKG": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.40,
        "IC Components": 0.45
    },
    "IDN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.40,
        "IC Components": 0.45
    },

    # -------------------------
    # Tier 3 — Low Semiconductor Presence
    # -------------------------
    "ARE": {
        "Microprocessors": 0.05,
        "Power Devices": 0.15,
        "Transistors": 0.55,
        "IC Components": 0.25
    },
    "AUS": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.55,
        "IC Components": 0.30
    },
    "BEL": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.40,
        "IC Components": 0.30
    },
    "BRA": {
        "Microprocessors": 0.05,
        "Power Devices": 0.20,
        "Transistors": 0.45,
        "IC Components": 0.30
    },
    "FIN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.40,
        "IC Components": 0.30
    }
}


In [10]:

# Function to sample a product for each supplier

rng = np.random.default_rng(SEED)

def assign_product(country_code):
    weights = country_product_weights[country_code]
    products = list(weights.keys())
    probs = np.array(list(weights.values()))
    return rng.choice(products, p=probs)


# Apply to suppliers DataFrame
suppliers["product"] = suppliers["country_code"].apply(assign_product)


In [11]:
suppliers.tail(30)

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product
61,MYS,SUP_MYS_62,93.674561,5.874361,34.508121,0.408347,0.611556,0.709132,Transistors
62,MYS,SUP_MYS_63,92.603276,6.587282,43.392282,0.368410,0.607717,0.738115,Transistors
63,MYS,SUP_MYS_64,96.626675,5.521518,30.487166,0.416490,0.616157,0.711582,Microprocessors
64,NLD,SUP_NLD_65,99.248126,8.955970,80.209394,0.199309,0.797345,0.843008,IC Components
65,NLD,SUP_NLD_66,87.501044,7.577642,57.420661,0.205113,0.825017,0.797405,IC Components
66,NLD,SUP_NLD_67,90.568945,8.214084,67.471181,0.225406,0.833884,0.787090,Microprocessors
67,NLD,SUP_NLD_68,96.742753,7.825076,61.231819,0.206893,0.880993,0.802450,Power Devices
68,SGP,SUP_SGP_69,103.816474,7.031540,49.442561,0.142956,0.861526,0.869948,IC Components
69,SGP,SUP_SGP_70,104.685820,5.770543,33.299169,0.129999,0.940477,0.847667,Power Devices
70,SGP,SUP_SGP_71,94.535090,5.595775,31.312693,0.137044,0.904991,0.919955,IC Components


# Creating a Probability of Defect column based on what the country behind the supplier *tends* to manufacture in the modern semiconductor industry

Criteria:
 - How aligned the supplier's assigned product is with what the country is actually known for producing
 - how mature and reliable that country's manufacturing base is
 - How complex the product category is (microprocessors are more difficult to manufacture; thus, higher defect risk)
 - Added a little noise for variation within each countries list of suppliers

In [12]:
# Baseline difficulty per product
product_difficulty = {
    "Microprocessors": 0.12,
    "Power Devices": 0.07,
    "Transistors": 0.05,
    "IC Components": 0.025
}





rng = np.random.default_rng(SEED)
# Function to compute probability_of_defect
def compute_defect_probability(country_code, product):
    # Get the country’s weight for this product
    weight = country_product_weights[country_code][product]

    # Alignment factor (higher = worse alignment)
    alignment_factor = 1 - weight

    # Baseline difficulty
    difficulty = product_difficulty[product]

    # Final defect probability
    defect_prob = difficulty * (0.5 + 0.5 * alignment_factor) * rng.uniform(0.9, 1.1)

    return defect_prob


In [13]:
# Apply to suppliers DataFrame
suppliers["probability_of_defect"] = suppliers.apply(
    lambda row: compute_defect_probability(row["country_code"], row["product"]),
    axis=1
)


In [14]:
suppliers.tail(30)

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect
61,MYS,SUP_MYS_62,93.674561,5.874361,34.508121,0.408347,0.611556,0.709132,Transistors,0.043439
62,MYS,SUP_MYS_63,92.603276,6.587282,43.392282,0.368410,0.607717,0.738115,Transistors,0.045000
63,MYS,SUP_MYS_64,96.626675,5.521518,30.487166,0.416490,0.616157,0.711582,Microprocessors,0.104731
64,NLD,SUP_NLD_65,99.248126,8.955970,80.209394,0.199309,0.797345,0.843008,IC Components,0.022970
65,NLD,SUP_NLD_66,87.501044,7.577642,57.420661,0.205113,0.825017,0.797405,IC Components,0.023331
66,NLD,SUP_NLD_67,90.568945,8.214084,67.471181,0.225406,0.833884,0.787090,Microprocessors,0.098469
67,NLD,SUP_NLD_68,96.742753,7.825076,61.231819,0.206893,0.880993,0.802450,Power Devices,0.059513
68,SGP,SUP_SGP_69,103.816474,7.031540,49.442561,0.142956,0.861526,0.869948,IC Components,0.019613
69,SGP,SUP_SGP_70,104.685820,5.770543,33.299169,0.129999,0.940477,0.847667,Power Devices,0.060659
70,SGP,SUP_SGP_71,94.535090,5.595775,31.312693,0.137044,0.904991,0.919955,IC Components,0.019001


# Adding Bulk-Order Discount Rate Column

Criteria:
- how mature and high-volume a country's semiconductor manufacturing base is
- how aligned the supplier's assigned product is with what that country is actually known for producing
- how competitive the country is in packaging markets
- how aggressively suppliers in that country typically discount for volume

In [15]:
rng = np.random.default_rng(SEED)
# ------------------------------------------------------------
# Base bulk discount by product (industry realistic)
# ------------------------------------------------------------
base_bulk_discount = {
    "Microprocessors": 0.04,   # 4%
    "Power Devices": 0.07,     # 7%
    "Transistors": 0.11,       # 11%
    "IC Components": 0.14      # 14%
}

# Function to compute bulk discount based on country-product alignment
def compute_bulk_discount(country_code, product):
    # Country's strength in this product
    weight = country_product_weights[country_code][product]

    # Base discount for this product category
    base = base_bulk_discount[product]

    # Final discount: stronger alignment → deeper discount
    discount = base * (0.3 + 0.7 * weight) * rng.uniform(0.95, 1.05)

    return discount


# Apply to suppliers DataFrame
suppliers["bulk_discount"] = suppliers.apply(
    lambda row: compute_bulk_discount(row["country_code"], row["product"]),
    axis=1
)


In [16]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount
0,ARE,SUP_ARE_1,127.022009,9.676976,93.643856,0.245273,0.661656,0.825186,Power Devices,0.060592,0.027440
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,Transistors,0.037264,0.076404
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.036013,0.075103
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,Transistors,0.035311,0.074374
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.035198,0.074257
...,...,...,...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,Transistors,0.044923,0.048359
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.044637,0.052780
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.044310,0.052585
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,IC Components,0.023583,0.069097


# Adding a bulk-units column i.e., bulk discounts only applied when tied to a minimum order quantity.

Depends on:
 - product category
 - country's manufacturing maturity
 - country's cost structure
 - country's alignment with that product (weight matrix)

Country alignment criteria:
 - If a country is strong in a product, MOQ is lower.
 - If weak, MOQ is higher.

In [17]:
# ------------------------------------------------------------
# Base MOQ by product
# ------------------------------------------------------------
rng = np.random.default_rng(SEED)

base_moq = { # typica MOQ before adjusting for country-product manufacturing strength
    "Microprocessors": 1000,
    "Power Devices": 2000,
    "Transistors": 5000,
    "IC Components": 8000
}


# Funtion to compute MOQ (bulk_units needed to activate discount rate for order) based on country-product alignment
def compute_bulk_units(country_code, product):
    weight = country_product_weights[country_code][product] # higher weight = more capable of efficiently producing
    base = base_moq[product] # e.g., microprocessors are a more high-end component. lower MOQ before discount sets in

    # Lower weight → higher MOQ (less efficient producers less eager to apply discount); higher weight (more efficient at producting) → lower MOQ (for discount to activate)
    multiplier = 1.15 - 0.6 * weight
    # multiplier larger if weight (efficiency of producing) is lower
    # multiplier lower if weight is larger

    moq = base * multiplier * rng.uniform(0.95, 1.05)

    # Ensure MOQ is at least 1
    return max(int(moq), 1)


# Apply to suppliers DataFrame
suppliers["bulk_units"] = suppliers.apply(
    lambda row: compute_bulk_units(row["country_code"], row["product"]),
    axis=1
)


In [18]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units
0,ARE,SUP_ARE_1,127.022009,9.676976,93.643856,0.245273,0.661656,0.825186,Power Devices,0.060592,0.027440,2051
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,Transistors,0.037264,0.076404,4157
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.036013,0.075103,4086
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,Transistors,0.035311,0.074374,4046
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.035198,0.074257,4040
...,...,...,...,...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,Transistors,0.044923,0.048359,5145
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.044637,0.052780,5050
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.044310,0.052585,5032
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,IC Components,0.023583,0.069097,8312


In [19]:
products

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.213043,integrated_circuit_components,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
...,...,...,...,...,...,...,...,...,...
13035,2025-12-01,BRA,FRED,OAC,98.700000,0.500652,power_devices,2025,12
13036,2025-12-01,IND,FRED,OAC,98.700000,0.450587,power_devices,2025,12
13037,2025-12-01,THA,FRED,OAC,98.700000,0.440574,power_devices,2025,12
13038,2025-12-01,IDN,FRED,OAC,98.700000,0.430561,power_devices,2025,12


# Creating a baseline price column

In [20]:
product_name_map = {
    "IC Components": "integrated_circuit_components",
    "Transistors": "transistors",
    "Power Devices": "power_devices",
    "Microprocessors": "microprocessors"
}


In [21]:
products.loc[lambda x: x['product'] == 'transistors']

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
6520,2012-06-01,USA,FRED,USA,99.797898,0.139717,transistors,2012,6
6521,2012-06-01,JPN,FRED,OAC,100.000000,0.142029,transistors,2012,6
6522,2012-06-01,DEU,FRED,USA,99.797898,0.139717,transistors,2012,6
6523,2012-06-01,FRA,FRED,USA,99.797898,0.139717,transistors,2012,6
6524,2012-06-01,GBR,FRED,USA,99.797898,0.139717,transistors,2012,6
...,...,...,...,...,...,...,...,...,...
9775,2025-12-01,BRA,FRED,OAC,98.700000,0.120157,transistors,2025,12
9776,2025-12-01,IND,FRED,OAC,98.700000,0.100130,transistors,2025,12
9777,2025-12-01,THA,FRED,OAC,98.700000,0.100130,transistors,2025,12
9778,2025-12-01,IDN,FRED,OAC,98.700000,0.100130,transistors,2025,12


In [22]:
# Make a copy of products with normalized product names
products_copy = products.copy()
products_copy["product_norm"] = products_copy["product"].str.lower().str.strip()

# Compute most recent real_price per (country_code, product)
latest_prices = (
    products_copy.sort_values("date")
    .groupby(["country_code", "product_norm"])["real_price"]
    .last()
    .reset_index()
)


In [23]:
# Normalize supplier product names to match products table
suppliers["product_norm"] = suppliers["product"].map(product_name_map)

# Merge baseline prices
suppliers = suppliers.merge(
    latest_prices,
    how="left",
    left_on=["country_code", "product_norm"],
    right_on=["country_code", "product_norm"]
)

# Rename for clarity
suppliers = suppliers.rename(columns={"real_price": "baseline_price"})


In [24]:
suppliers = suppliers.drop(columns=["product_norm"])


In [25]:
suppliers.drop(suppliers.loc[lambda x: x['baseline_price'].isna()].index, axis=0, inplace=True)

In [26]:
suppliers.loc[lambda x: x['country_code']== 'THA']

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
78,THA,SUP_THA_79,100.277387,5.412812,29.298529,0.489983,0.477038,0.710978,Microprocessors,0.105853,0.012762,1066,0.821070
79,THA,SUP_THA_80,98.681799,6.505389,42.320083,0.475889,0.445450,0.690970,Power Devices,0.058772,0.032577,1959,0.440574
80,THA,SUP_THA_81,100.475495,5.644594,31.861442,0.446817,0.487221,0.682779,Power Devices,0.058575,0.032524,1956,0.440574
81,THA,SUP_THA_82,96.224869,6.099137,37.199468,0.467973,0.483340,0.686891,IC Components,0.019394,0.092564,6916,0.140183
82,THA,SUP_THA_83,98.083843,5.424881,29.429330,0.498967,0.518076,0.725676,Transistors,0.044886,0.048339,5143,0.100130
83,THA,SUP_THA_84,96.502208,5.437766,29.569298,0.480340,0.485508,0.704155,Power Devices,0.056263,0.031896,1918,0.440574
84,THA,SUP_THA_85,95.701557,6.085417,37.032304,0.475536,0.478193,0.702526,Microprocessors,0.105601,0.012747,1065,0.821070
85,THA,SUP_THA_86,98.579174,5.754680,33.116337,0.430113,0.518949,0.696473,IC Components,0.019141,0.091949,6870,0.140183
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,Transistors,0.044923,0.048359,5145,0.100130


In [27]:
suppliers = suppliers.assign(
    bulk_units = lambda x: x.bulk_units.astype(int)
)

In [28]:
suppliers.tail()

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,Transistors,0.044923,0.048359,5145,0.100130
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.044637,0.052780,5050,0.131908
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.044310,0.052585,5032,0.131908
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,IC Components,0.023583,0.069097,8312,0.188440
90,USA,SUP_USA_91,66.503117,5.185243,26.886741,0.344072,0.729554,0.803135,IC Components,0.023705,0.069282,8334,0.188440


# Injecting Some Noise into Baseline Prices by Supplier

In [29]:
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# Product-specific noise ranges
# ------------------------------------------------------------
product_noise_ranges = {
    "Microprocessors": 0.03,   # +/-3%
    "Power Devices": 0.05,     # +/-5%
    "Transistors": 0.08,       # +/-8%
    "IC Components": 0.10      # +/-10%
}


# Pre-generate country-level cost factors (ONE per country)
country_cost_factors = {
    country: rng.uniform(0.95, 1.05)
    for country in country_product_weights.keys()
}


# Function to apply structured noise to each suppliers baseprice for controlled differentation
def add_product_specific_noise(row):
    base_price = row["baseline_price"]
    product = row["product"]
    country = row["country_code"]

    # Product-level variability
    noise_scale = product_noise_ranges[product]

    # Supplier-level variation (small)
    supplier_noise = rng.uniform(1 - noise_scale, 1 + noise_scale)

    # Country-level systematic effect (fixed per country)
    country_factor = country_cost_factors[country]

    # Final price
    return base_price * country_factor * supplier_noise

In [30]:
# Apply to suppliers DataFrame
suppliers["baseline_price"] = suppliers.apply(add_product_specific_noise, axis=1)

In [31]:
suppliers.tail()

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,Transistors,0.044923,0.048359,5145,0.101508
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.044637,0.052780,5050,0.137421
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.044310,0.052585,5032,0.132664
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,IC Components,0.023583,0.069097,8312,0.179070
90,USA,SUP_USA_91,66.503117,5.185243,26.886741,0.344072,0.729554,0.803135,IC Components,0.023705,0.069282,8334,0.191492


In [32]:
products

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product,year,month
0,2012-06-01,USA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
1,2012-06-01,JPN,FRED,OAC,100.000000,0.213043,integrated_circuit_components,2012,6
2,2012-06-01,DEU,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
3,2012-06-01,FRA,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
4,2012-06-01,GBR,FRED,USA,99.797898,0.199596,integrated_circuit_components,2012,6
...,...,...,...,...,...,...,...,...,...
13035,2025-12-01,BRA,FRED,OAC,98.700000,0.500652,power_devices,2025,12
13036,2025-12-01,IND,FRED,OAC,98.700000,0.450587,power_devices,2025,12
13037,2025-12-01,THA,FRED,OAC,98.700000,0.440574,power_devices,2025,12
13038,2025-12-01,IDN,FRED,OAC,98.700000,0.430561,power_devices,2025,12


# Creating a Price Volatility for each supplier based on their disruption probability and rolling standard deviation of real_price of product (grouped by country in products table) over last 5 years

In [33]:
product_map = {
    "IC Components": "integrated_circuit_components",
    "Transistors": "transistors",
    "Power Devices": "power_devices",
    "Microprocessors": "microprocessors"
}

suppliers["product_norm"] = suppliers["product"].map(product_map)
products["product_norm"] = products["product"].str.lower().str.strip()


In [34]:
products_sorted = products.sort_values(["country_code", "product_norm", "date"])

products_sorted["price_volatility_raw"] = (
    products_sorted
    .groupby(["country_code", "product_norm"])["real_price"]
    .rolling(60, min_periods=12)
    .std()
    .reset_index(level=[0,1], drop=True)
)


In [35]:
latest_volatility = (
    products_sorted
    .groupby(["country_code", "product_norm"])["price_volatility_raw"]
    .last()
    .reset_index()
)


In [36]:
suppliers = suppliers.merge(
    latest_volatility,
    how="left",
    left_on=["country_code", "product_norm"],
    right_on=["country_code", "product_norm"]
)


In [37]:
min_v = suppliers["price_volatility_raw"].min()
max_v = suppliers["price_volatility_raw"].max()

suppliers["price_volatility_norm"] = (
    (suppliers["price_volatility_raw"] - min_v) / (max_v - min_v)
)


In [38]:
suppliers["price_volatility"] = (
    0.6 * suppliers["price_volatility_norm"] +
    0.4 * suppliers["disruption_probability"]
)


In [39]:
suppliers = suppliers.drop(columns=["price_volatility_raw", "price_volatility_norm",
                                    'product_norm'])


In [40]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility
0,ARE,SUP_ARE_1,127.022009,9.676976,93.643856,0.245273,0.661656,0.825186,Power Devices,0.060592,0.027440,2051,0.580499,0.349750
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,Transistors,0.037264,0.076404,4157,0.128066,0.139689
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.036013,0.075103,4086,0.132879,0.134274
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,Transistors,0.035311,0.074374,4046,0.119404,0.134306
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.035198,0.074257,4040,0.128414,0.140060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,Transistors,0.044923,0.048359,5145,0.101508,0.222735
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.044637,0.052780,5050,0.137421,0.148080
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.044310,0.052585,5032,0.132664,0.127469
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,IC Components,0.023583,0.069097,8312,0.179070,0.155067


In [41]:
suppliers.loc[lambda x: x['product'] == 'Transistors']

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,Transistors,0.037264,0.076404,4157,0.128066,0.139689
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.036013,0.075103,4086,0.132879,0.134274
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,Transistors,0.035311,0.074374,4046,0.119404,0.134306
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.035198,0.074257,4040,0.128414,0.140060
9,CAN,SUP_CAN_10,71.551361,7.364183,54.231184,0.217086,0.885767,0.803172,Transistors,0.039586,0.058741,4605,0.112801,0.086834
14,CHN,SUP_CHN_15,95.289012,5.551037,30.814013,0.459787,0.467951,0.773362,Transistors,0.043883,0.052329,5007,0.069323,0.187197
16,CHN,SUP_CHN_17,94.110556,6.031509,36.379098,0.397835,0.500291,0.760977,Transistors,0.043298,0.051980,4974,0.071806,0.162416
17,CHN,SUP_CHN_18,97.153916,5.332108,28.431372,0.440220,0.449032,0.748863,Transistors,0.042340,0.051408,4919,0.070320,0.179370
22,DEU,SUP_DEU_23,87.354670,4.723451,22.310992,0.242089,0.725685,0.851682,Transistors,0.045178,0.053103,5081,0.132016,0.108094
25,FIN,SUP_FIN_26,111.561665,10.250021,105.062934,0.178654,0.918622,0.800042,Transistors,0.038921,0.062940,4488,0.128157,0.082720


# Applying Tariff Code to Products if Foreign Supplier

In [42]:
tariff = pd.read_csv('/content/tarriff_database_2025_only_semiconductor_components_v2.csv')

In [43]:
tariff.loc[lambda x: x.mfn_text_rate_pct > 0]

,hts8,brief_description,mfn_text_rate_pct,how_measured
10,84807180,"Molds for rubber or plastics, injection or com...",3.1,NO
14,84879000,"Machinery parts, not containing electrical con...",3.9,KG
17,85042300,Liquid dielectric transformers having a power ...,1.6,NO
19,85043140,Electrical transformers other than liquid diel...,6.6,NO
20,85043160,Electrical transformers other than liquid diel...,1.6,NO
21,85043200,Electrical transformers other than liquid diel...,2.4,NO
22,85043300,Electrical transformers other than liquid diel...,1.6,NO
23,85043400,Electrical transformers other than liquid diel...,1.6,NO
31,85118020,Voltage and voltage-current regulators with cu...,2.5,NO
33,85119020,Parts of voltage and voltage-current regulator...,3.1,KG


Code 85389030 can potentially be included under IC components

In [44]:
suppliers = suppliers.assign(
    hts8_tariff = lambda x: np.where(
        (x['product'] == 'IC Components') & (x['country_code'] != 'USA'), '85389030', 'None')
)

In [45]:
suppliers.to_csv('suppliers_products_UPDATED.csv', index=False)

In [50]:
suppliers.loc[lambda x: x['product'] == 'IC Components',
              'probability_of_defect':'hts8_tariff'].tail(30)

,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility,hts8_tariff
10,0.021868,0.084992,7619,0.151972,0.092811,85389030
11,0.021002,0.089715,7335,0.101881,0.179366,85389030
12,0.019901,0.087270,7135,0.100418,0.179513,85389030
13,0.020354,0.088276,7217,0.096803,0.185816,85389030
15,0.020638,0.088906,7269,0.103250,0.182015,85389030
31,0.022842,0.074075,8050,0.191330,0.137485,85389030
32,0.020528,0.088661,7249,0.171738,0.162958,85389030
33,0.019788,0.087019,7115,0.174343,0.171187,85389030
35,0.021106,0.089947,7354,0.180654,0.163556,85389030
36,0.019621,0.086646,7084,0.177218,0.164379,85389030
